# Data utilities

> Load datasets by path, cache name or Hugging Face repo; convert, merge and normalize them (stable-worldmodel API).

In [ ]:
#| default_exp data.utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import json
import logging
import urllib.request
from pathlib import Path
from typing import Any

import numpy as np

from stable_marl.data.dataset import Dataset, get_cache_dir
from stable_marl.data.format import EPISODE_DATA_KEY, FORMATS, detect_format, get_format
from stable_marl.utils.core import HF_BASE_URL
from stable_marl.data.normalization import (IdentityScaler, PercentileScaler, ZScoreScaler, ColumnTransform,
                                            get_scaler)

logger = logging.getLogger(__name__)

## Loading

`load_dataset(name)` accepts, in order: a scheme-prefixed id (`scheme://...`, given to the format
that detects it), a path (absolute, or relative to `<cache dir>/datasets`; `name.h5` is tried too),
or a Hugging Face dataset repo `<user>/<repo>`, downloaded once into
`<cache dir>/datasets/<user>--<repo>/`. The cache dir is `$STABLEMARL_HOME`, else `~/.stable_marl`.

In [ ]:
#| export
def ensure_dir_exists(path: Path):
    Path(path).mkdir(parents=True, exist_ok=True)


def load_dataset(name: str | Path, cache_dir: str | None = None, format: str | None = None, **kwargs: Any) -> Dataset:
    "Resolve `name` to a dataset and open it with its (detected, or given) format's reader, passing `kwargs`."
    name = str(name)
    if '://' in name:
        if format is not None:
            return get_format(format).open_reader(name, **kwargs)
        for fmt in FORMATS.values():
            if fmt.detect(name):
                return fmt.open_reader(name, **kwargs)
        raise ValueError(f"No format detected for {name!r}")
    path = _resolve_dataset(name, get_cache_dir(cache_dir, sub_folder='datasets'))
    if format is not None:
        return get_format(format).open_reader(path, **kwargs)
    fmt = detect_format(path)
    if fmt is None:
        raise ValueError(f"No format detected for {path!r}; pass format= explicitly.")
    return fmt.open_reader(path, **kwargs)


def _resolve_dataset(name: str, datasets_dir: Path) -> Path:
    "A local path for `name` (a path, a name in `datasets_dir`, or a Hugging Face repo id, downloaded)."
    local = Path(name).expanduser()
    if not local.is_absolute():
        local = datasets_dir / local
    for candidate in (local, local.with_name(local.name + '.h5')):
        if candidate.exists():
            return candidate
    if '/' in name and not name.startswith(('.', '/', '~')):
        return _resolve_dataset_hf(name, datasets_dir)
    raise FileNotFoundError(f"Cannot resolve {name!r}: not a local path or HF repo id.")


_HF_FILE_SUFFIXES = ('.h5', '.hdf5')
_HF_DIR_SUFFIXES = ('.lance',)


def _hf_list_tree(repo_id: str, sub_path: str = '') -> list[dict]:
    "The entries at ``<repo>/tree/main/<sub_path>`` (one Hugging Face API call)."
    url = f"{HF_BASE_URL}/api/datasets/{repo_id}/tree/main{'/' + sub_path if sub_path else ''}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())


def _hf_find_dataset_entry(repo_id: str) -> dict:
    "The first top-level dataset entry: a ``*.lance`` folder, else a ``*.h5`` / ``*.hdf5`` file."
    entries = _hf_list_tree(repo_id)
    for kind, suffixes in (('directory', _HF_DIR_SUFFIXES), ('file', _HF_FILE_SUFFIXES)):
        for entry in entries:
            if entry.get('type') == kind and entry.get('path', '').endswith(suffixes):
                return entry
    raise FileNotFoundError(f"No dataset found in HF repo {repo_id}: expected a top-level `*.lance` "
                            "directory or `*.h5` / `*.hdf5` file.")


def _hf_walk_files(repo_id: str, sub_path: str) -> list[str]:
    "Every file under `sub_path`."
    out, stack = [], [sub_path]
    while stack:
        for entry in _hf_list_tree(repo_id, stack.pop()):
            (stack if entry.get('type') == 'directory' else out).append(entry.get('path', ''))
    return out


def _resolve_dataset_hf(repo_id: str, datasets_dir: Path) -> Path:
    "The local copy of a Hugging Face dataset repo, downloaded on first use."
    local_dir = datasets_dir / repo_id.replace('/', '--')
    if local_dir.is_dir() and any(local_dir.iterdir()):
        logger.info(f"Using cached dataset for {repo_id} at {local_dir}")
        return local_dir
    logger.info(f"Downloading dataset {repo_id} from Hugging Face...")
    local_dir.mkdir(parents=True, exist_ok=True)
    entry = _hf_find_dataset_entry(repo_id)
    remotes = _hf_walk_files(repo_id, entry['path']) if entry.get('type') == 'directory' else [entry['path']]
    if not remotes:
        raise FileNotFoundError(f"HF repo {repo_id}: directory '{entry['path']}' is empty.")
    for remote in remotes:
        dest = local_dir / remote
        dest.parent.mkdir(parents=True, exist_ok=True)
        _download(f"{HF_BASE_URL}/datasets/{repo_id}/resolve/main/{remote}", dest)
    return local_dir


def _download(url: str, dest: Path) -> None:
    "Download `url` to `dest` (with a progress bar if tqdm is installed)."
    response = urllib.request.urlopen(url)
    total = int(response.headers.get('Content-Length', 0)) or None
    try:
        from tqdm import tqdm
        bar = tqdm(total=total, unit='B', unit_scale=True, desc=Path(dest).name)
    except ImportError:
        bar = None
    with open(dest, 'wb') as f:
        while chunk := response.read(1 << 16):
            f.write(chunk)
            if bar is not None: bar.update(len(chunk))
    if bar is not None: bar.close()

## Converting and merging

`convert` rewrites a dataset in another format, and `merge` writes several datasets (same columns)
into one, episodes renumbered in order. Episodes are read as stored when the reader gives raw rows
(`get_row_data`, as `HDF5Dataset` does): every column round-trips exactly, strings included.
Other readers go through `load_episode`, as in stable-worldmodel: channel-first images are put back
channel-last and float images rescaled to `uint8`.

In [ ]:
#| export
def _episode_steps(src, ep_idx: int) -> dict[str, list]:
    "Episode `ep_idx` of `src` as the ``{column: [row, ...]}`` writers take."
    length = int(src.lengths[ep_idx])
    if hasattr(src, 'offsets') and type(src).get_row_data is not Dataset.get_row_data:
        start = int(src.offsets[ep_idx])
        rows = src.get_row_data(list(range(start, start + length)))
        return {col: list(np.asarray(rows[col])) for col in src.column_names}
    return _episode_to_step_lists(src.load_episode(ep_idx), length)


def _episode_to_step_lists(ep: dict, ep_len: int) -> dict[str, list]:
    """
    A reader's episode (tensors) as ``{column: [row, ...]}``: channel-first images
    ``(T, C, H, W)`` back to ``(T, H, W, C)``, float images (``(T, H, W, C)`` or per agent
    ``(T, num_agents, H, W, C)``, values in ``[0, 1]``) to ``uint8``, scalars repeated.
    """
    out = {}
    for col, val in ep.items():
        if type(val).__module__.startswith('torch'):
            arr = val.detach().cpu().numpy()
        elif isinstance(val, np.ndarray):
            arr = val
        else:
            out[col] = [val] * ep_len
            continue
        if arr.ndim == 4 and arr.shape[1] in (1, 3):
            arr = arr.transpose(0, 2, 3, 1)
        if arr.dtype.kind == 'f' and arr.ndim in (4, 5) and arr.shape[-1] in (1, 3):
            arr = (np.clip(arr, 0.0, 1.0) * 255.0).astype(np.uint8)
        out[col] = list(arr)
    return out


def _with_episode_data(src, ep_idx: int, steps: dict, carry: bool) -> dict:
    if carry:
        steps[EPISODE_DATA_KEY] = {k: v[0] for k, v in src.get_episode_data([ep_idx]).items()}
    return steps


def _progress(it, desc: str, progress: bool):
    if not progress:
        return it
    try:
        from tqdm import tqdm
        return tqdm(it, desc=desc)
    except ImportError:
        return it


def convert(
    source: str | Path,
    dest: str | Path,
    *,
    source_format: str | None = None,
    dest_format: str = 'hdf5',
    cache_dir: str | None = None,
    progress: bool = True,
    **dest_kwargs: Any) -> None:
    """
    Rewrite the dataset `source` (anything :func:`load_dataset` takes) at `dest` in `dest_format`,
    episode by episode; `dest_kwargs` go to the writer (e.g. ``mode``, ``compression``).
    Episode-scoped data is kept if the destination format supports it.
    """
    src = load_dataset(source, cache_dir=cache_dir, format=source_format)
    writer_cls = get_format(dest_format)
    ep_cols = list(getattr(src, 'episode_column_names', None) or [])
    carry = bool(ep_cols) and getattr(writer_cls, 'supports_episode_data', False)
    if ep_cols and not carry:
        logger.warning(f"convert: '{dest_format}' does not support episode data; dropping {ep_cols}.")
    episodes = (_with_episode_data(src, i, _episode_steps(src, i), carry)
                for i in _progress(range(len(src.lengths)), f'Converting -> {dest_format}', progress))
    with writer_cls.open_writer(dest, **dest_kwargs) as writer:
        writer.write_episodes(episodes)


def merge(
    sources,
    dest: str | Path,
    *,
    dest_format: str = 'hdf5',
    source_formats: list[str | None] | None = None,
    mode: str = 'error',
    cache_dir: str | None = None,
    progress: bool = True,
    **dest_kwargs) -> None:
    """
    Write the episodes of every source, in order, into one dataset at `dest` (episodes renumbered
    ``0 .. N-1``). The sources must have the same columns (checked before anything is written).
    `mode`: ``'error'`` if `dest` exists (default), ``'overwrite'`` or ``'append'``.
    """
    sources = list(sources)
    if len(sources) < 2:
        raise ValueError("merge needs at least two source datasets")
    if source_formats is not None and len(source_formats) != len(sources):
        raise ValueError("source_formats must align with sources")
    fmt = lambda i: source_formats[i] if source_formats else None
    writer_cls = get_format(dest_format)

    reference = ep_reference = ref_source = None
    for i, source in enumerate(sources):
        ds = load_dataset(source, cache_dir=cache_dir, format=fmt(i))
        cols, ep_cols = set(ds.column_names), set(getattr(ds, 'episode_column_names', None) or [])
        if reference is None:
            reference, ep_reference, ref_source = cols, ep_cols, source
            continue
        if cols != reference:
            raise ValueError(f"merge: column mismatch between {ref_source!r} and {source!r}: missing "
                             f"{sorted(reference - cols)}; unexpected {sorted(cols - reference)}. "
                             "All sources must share the same columns.")
        if ep_cols != ep_reference:
            raise ValueError(f"merge: episode-data column mismatch between {ref_source!r} and {source!r}: missing "
                             f"{sorted(ep_reference - ep_cols)}; unexpected {sorted(ep_cols - ep_reference)}.")
    carry = bool(ep_reference) and getattr(writer_cls, 'supports_episode_data', False)
    if ep_reference and not carry:
        logger.warning(f"merge: '{dest_format}' does not support episode data; dropping {sorted(ep_reference)}.")

    def episodes():
        for i, source in enumerate(sources):
            src = load_dataset(source, cache_dir=cache_dir, format=fmt(i))
            for ep_idx in _progress(range(len(src.lengths)), f'Merging {source}', progress):
                yield _with_episode_data(src, ep_idx, _episode_steps(src, ep_idx), carry)
            if hasattr(src, 'close'):
                src.close()

    with writer_cls.open_writer(dest, mode=mode, **dest_kwargs) as writer:
        writer.write_episodes(episodes())

## Normalization

In [ ]:
#| export
def column_normalizer(dataset: Dataset, source: str, target: str, method: str = 'zscore') -> ColumnTransform:
    """
    A dataset ``transform`` normalizing column `source` into `target`, with a scaler
    (``'zscore'``, ``'percentile'`` or ``'none'``) fitted on the whole column.
    """
    scaler = get_scaler(method)
    if method != 'none':
        scaler.fit(np.array(dataset.get_col_data(source)))
    return ColumnTransform(scaler, source=source, target=target)

### Tests

In [ ]:
import tempfile, shutil, torch
from unittest import mock
from fastcore.test import test_fail
import stable_marl as sm
from stable_marl.data import HDF5Dataset, HDF5Writer, ConcatDataset

def collect(path, seed):
    world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=2, agents=3, size=7, num_obstacles=0, n_clutter=0,
                     min_goal_spawn_distance=1, tile_size=4, goal_conditioned=True, max_episode_steps=8)
    world.set_policy(sm.RandomPolicy(seed=seed))
    world.collect(path, episodes=3, seed=seed, progress=False)

def columns(path):
    with h5py.File(path, 'r') as f:
        return {k: f[k][:] for k in f}

import h5py
with tempfile.TemporaryDirectory() as tmp:
    collect(f'{tmp}/a.h5', 0); collect(f'{tmp}/b.h5', 10)

    # load_dataset: a path, a name in the cache dir (with or without .h5), keyword arguments for the reader
    assert len(load_dataset(f'{tmp}/a.h5').lengths) == 3
    shutil.copy(f'{tmp}/a.h5', get_cache_dir(tmp, 'datasets') / 'cached.h5')
    ds = load_dataset('cached', cache_dir=tmp, num_steps=2, keys_to_load=['pixels', 'action'])
    assert ds[0]['pixels'].shape == (2, 3, 28, 28, 3) and ds.column_names == ['pixels', 'action']
    assert load_dataset('cached.h5', cache_dir=tmp, format='hdf5').num_steps == 1
    test_fail(lambda: load_dataset('nothing_here', cache_dir=tmp), contains='Cannot resolve')

    # a Hugging Face repo (download faked): fetched once into <cache>/datasets/user--repo/
    tree = {'': [{'type': 'file', 'path': 'README.md'}, {'type': 'file', 'path': 'findgoal.h5'}]}
    fake_download = mock.Mock(side_effect=lambda url, dest: shutil.copy(f'{tmp}/a.h5', dest))
    saved = _hf_list_tree, _download                      # the notebook's own functions, faked here
    _hf_list_tree = lambda repo, sub='': tree[sub]
    _download = fake_download
    try:
        hf = load_dataset('someone/findgoal', cache_dir=tmp)
        again = load_dataset('someone/findgoal', cache_dir=tmp)
    finally:
        _hf_list_tree, _download = saved
    assert fake_download.call_count == 1 and fake_download.call_args[0][0].endswith('/datasets/someone/findgoal/resolve/main/findgoal.h5')
    assert hf.lengths.tolist() == again.lengths.tolist() == HDF5Dataset(path=f'{tmp}/a.h5').lengths.tolist()

    # convert: an exact copy (other compression)
    convert(f'{tmp}/a.h5', f'{tmp}/a_lzf.h5', compression='lzf', progress=False)
    a, a2 = columns(f'{tmp}/a.h5'), columns(f'{tmp}/a_lzf.h5')
    assert a.keys() == a2.keys() and all(np.array_equal(a[k], a2[k], equal_nan=a[k].dtype.kind == 'f') for k in a)

    # merge: the episodes of a then b, column by column identical to both
    merge([f'{tmp}/a.h5', f'{tmp}/b.h5'], f'{tmp}/ab.h5', progress=False)
    b, ab = columns(f'{tmp}/b.h5'), columns(f'{tmp}/ab.h5')
    for k in a:
        if k in ('ep_len', 'ep_offset'): continue
        assert np.array_equal(ab[k], np.concatenate([a[k], b[k]]), equal_nan=a[k].dtype.kind == 'f'), k
    assert ab['ep_len'].tolist() == a['ep_len'].tolist() + b['ep_len'].tolist()
    assert ab['ep_offset'].tolist() == list(np.cumsum([0, *ab['ep_len'][:-1]]))
    merged, concat = HDF5Dataset(path=f'{tmp}/ab.h5', num_steps=3), ConcatDataset([HDF5Dataset(path=f'{tmp}/{n}.h5', num_steps=3) for n in 'ab'])
    assert len(merged) == len(concat) and torch.equal(merged[len(merged) - 1]['pixels'], concat[len(concat) - 1]['pixels'])
    test_fail(lambda: merge([f'{tmp}/a.h5', f'{tmp}/b.h5'], f'{tmp}/ab.h5', progress=False), contains='already exists')
    test_fail(lambda: merge([f'{tmp}/a.h5'], f'{tmp}/x.h5'), contains='at least two')
    with HDF5Writer(f'{tmp}/other.h5') as w:
        w.write_episode({'pixels': np.zeros((2, 3, 1, 1, 3), np.uint8)})
    test_fail(lambda: merge([f'{tmp}/a.h5', f'{tmp}/other.h5'], f'{tmp}/y.h5', progress=False), contains='column mismatch')

    # column_normalizer: zscore of each agent's position, as a dataset transform
    norm = column_normalizer(HDF5Dataset(path=f'{tmp}/ab.h5'), 'position', 'position_norm')
    ds = HDF5Dataset(path=f'{tmp}/ab.h5', num_steps=2, transform=norm)
    item = ds[0]
    assert item['position_norm'].shape == item['position'].shape == (2, 3, 2) and item['position_norm'].dtype == torch.float32
    all_norm = norm.fn(torch.as_tensor(HDF5Dataset(path=f'{tmp}/ab.h5').get_col_data('position')).float())
    assert torch.allclose(all_norm.reshape(-1, 2).mean(0), torch.zeros(2), atol=1e-5)
    assert isinstance(column_normalizer(ds, 'position', 'p', method='none').fn, IdentityScaler)

# a reader without raw rows: images through load_episode (channel-first and float images back to uint8 HWC)
ep = {'pixels': torch.rand(4, 3, 5, 6), 'views': torch.rand(4, 2, 5, 6, 3), 'action': torch.zeros(4, 2, 3), 'task': 'go'}
steps = _episode_to_step_lists(ep, 4)
assert steps['pixels'][0].shape == (5, 6, 3) and steps['pixels'][0].dtype == np.uint8
assert steps['views'][0].dtype == np.uint8 and steps['action'][0].dtype == np.float32   # (agents, 3) actions stay float
assert steps['task'] == ['go'] * 4

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()